In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/data/VIN/ML/EL4TF


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
from torch.utils.data import DataLoader, TensorDataset


import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_multi_class_ml_tam import preprocess, TARGETS
from models.lstm import LSTMClassifier

In [2]:


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.svm import SVC
from loaders._load_vn30_multi_class_ml_tam import preprocess, TARGETS
from loaders._load_vn30_meta import VN30


In [4]:

# --- Tiền xử lý dữ liệu ---
symbol = 'VIC'  # Thay bằng mã cổ phiếu bạn muốn
X_train, X_test, y_train, y_test, scaler, classes = preprocess(symbol, verbose=True, val=0.1, use_scaler=True)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# --- Khởi tạo và huấn luyện mô hình SVM ---
model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
model.fit(X_train, y_train)

# --- Dự đoán và đánh giá ---
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"--- Evaluation for {symbol} ---")
print(classification_report(y_test, y_pred, target_names=classes, zero_division=0))
print(f"Overall Test Accuracy: {accuracy:.2f}%")
print(f"Overall Test Weighted F1-Score: {f1:.4f}")


=== Preprocessing VIC ===
Train: (1098, 120), Test: (420, 120)
Train shape: (1098, 120), Test shape: (420, 120)
--- Evaluation for VIC ---
              precision    recall  f1-score   support

 strong_down       0.00      0.00      0.00        38
   weak_down       0.00      0.00      0.00        78
    sideways       0.45      1.00      0.63       191
     weak_up       0.00      0.00      0.00        66
   strong_up       0.00      0.00      0.00        47

    accuracy                           0.45       420
   macro avg       0.09      0.20      0.13       420
weighted avg       0.21      0.45      0.28       420

Overall Test Accuracy: 45.48%
Overall Test Weighted F1-Score: 0.2843


In [7]:
# --- Đánh giá cho toàn bộ VN30 ---
eval_dict = {}
for symbol in VN30:
    X_train, X_test, y_train, y_test, scaler, classes = preprocess(symbol, verbose=False, val=0.1, use_scaler=True)
    model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred) * 100
    f1 = f1_score(y_test, y_pred, average='weighted')
    eval_dict[symbol] = (accuracy, f1)

# --- In top 5 mã có accuracy cao nhất ---
top5 = sorted(eval_dict.items(), key=lambda x: x[1][0], reverse=True)[:5]
print("\n--- Top 5 mã VN30 có accuracy cao nhất ---")
for symbol, (accuracy, f1) in top5:
    print(f"{symbol}: Accuracy = {accuracy:.2f}%, Weighted F1-Score = {f1:.4f}")


--- Top 5 mã VN30 có accuracy cao nhất ---
SSB: Accuracy = 61.26%, Weighted F1-Score = 0.4655
VJC: Accuracy = 50.71%, Weighted F1-Score = 0.3413
GAS: Accuracy = 46.90%, Weighted F1-Score = 0.3301
VIC: Accuracy = 45.48%, Weighted F1-Score = 0.2843
ACB: Accuracy = 44.63%, Weighted F1-Score = 0.2754
